<a href="https://colab.research.google.com/github/rgopereira/OAC/blob/main/IntroducaoMIPSColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assembly MIPS no Google Colab
**Objetivos da aula**
- Rodar programas em assembly MIPS32 direto no navegador, sem instalar nada
- Entender registradores, syscalls e o ciclo busca–decodificação–execução na prática
- Conhecer as duas ferramentas clássicas: SPIM e MARS - aqui iremos optar pelo MARS, tendo em vista diversos problemas para compilar um código no SPIM

> Tudo roda via linha de comando — o Colab não tem interface gráfica, e isso é uma
> vantagem: a saída vira texto e pode ser automatizada.

In [45]:
# ============================================================
# CÉLULA DE SETUP — rode no topo do notebook, uma vez por sessão
# A VM do Colab é efêmera: Java e Mars.jar somem a cada reinício
# ============================================================
!apt-get install -y default-jre-headless > /dev/null 2>&1
!test -f Mars.jar || wget -q https://dpetersanderson.github.io/Mars4_5.jar -O Mars.jar
!ls -la Mars.jar    # confirma o download (~1,4 MB)

-rw-r--r-- 1 root root 4169142 Feb  6  2025 Mars.jar


In [46]:
# VERIFICAÇÃO RÁPIDA DA INTEGRIDADE DO MARS
!java -version 2>&1 | head -1
!java -jar Mars.jar h 2>&1 | head -5    # imprime o help — o MARS respondeu

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
MARS 4.5  Copyright 2003-2014 Pete Sanderson and Kenneth Vollmar

Usage:  Mars  [options] filename [additional filenames]


## "Anatomia" de um Programa MIPS
Um programa tem duas seções ou segmentos: `.data` (variáveis estáticas) e `.text` (código do programador) - aqui iremos falar muito as palavras SEGMENTOS DE DADOS e SEGMENTO DE TEXTO.


A comunicação com o "sistema operacional" acontece por **syscalls**: colocamos o número do que chamamos de serviço em `$v0`, os argumentos que devem ser passados neste serviço, em `$a0`, e chamamos o serviço `syscall`.

| Registrador | Uso |
|---|---|
| `$zero` | sempre vale 0 |
| `$v0`–`$v1` | retorno / número da syscall |
| `$a0`–`$a3` | argumentos |
| `$t0`–`$t9` | temporários (podem ser sobrescritos) |
| `$s0`–`$s7` | preservados entre chamadas |
| `$sp`, `$ra` | pilha e endereço de retorno |

| `$v0` | Serviço | Entrada | Saída |
|---|---|---|---|
| 1 | imprimir inteiro | `$a0` | — |
| 4 | imprimir string | `$a0` = endereço | — |
| 5 | ler inteiro | — | `$v0` |
| 10 | encerrar | — | — |
| 11 | imprimir caractere | `$a0` | — |


Observe que no serviço do tipo 4 em $a0 é passado o endereço da string que deve ser impressa - então fique atento ao nome dado no segmento de dados (.data) e como você vai manipular essa texto - tenha em mente que você vai ter que lidar com o endereço inicial desta string (que pode ser um vetor).

Observe o código a seguir:

In [47]:
%%writefile hello.s
# hello.s — primeiro programa MIPS
        .data
msg:    .asciiz "Ola, MIPS 4.5 no Colab!\n"

        .text
        .globl main
main:
        li   $v0, 4        # syscall 4: imprimir string
        la   $a0, msg      # endereco da string em $a0
        syscall
        li   $v0, 10       # syscall 10: encerrar
        syscall

Overwriting hello.s


Observe que ele possui o que foi discutido anteriormente - segmento de dados e de texto. Agora vamos executar esse programa.
Compile utilizando o MARS 4.5.

In [48]:
# Monta e executa. Dica: depois de editar o hello.s, rode esta célula de novo.
!cat hello.s
!timeout 60 java -jar Mars.jar nc sm hello.s

# hello.s — primeiro programa MIPS
        .data
msg:    .asciiz "Ola, MIPS 4.5 no Colab!\n"

        .text
        .globl main
main:
        li   $v0, 4        # syscall 4: imprimir string
        la   $a0, msg      # endereco da string em $a0
        syscall
        li   $v0, 10       # syscall 10: encerrar
        syscall
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Ola, MIPS 4.5 no Colab!



O resultado deve ser algo como:

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

**Ola, MIPS 4.5 no Colab!** *texto em itálico*

Observe a mensagem escrita no segmento de dados foi impressa. Observe também o tipo da mensagem é do tipo **.asciiz**.



## 🎯 O "z" do .asciiz
Lembre-se:
- `.asciiz` = **ASCII + Z**ero: grava a string e um byte 0 (`\0`) no final
- Esse byte 0 é o **terminador nulo** — é assim que a syscall 4 sabe onde a string acaba
- `.ascii` grava só os caracteres, **sem** o 0

Agora vamos inserir **entrada** de dados - como se fosse um teclado. Considere o código abaixo, leia com atenção:

In [49]:
%%writefile soma.s
# soma.s — le dois inteiros e imprime a soma
        .text
        .globl main
main:
        li   $v0, 5        # syscall 5: ler inteiro -> $v0
        syscall
        move $t0, $v0      # salva o primeiro em $t0
        li   $v0, 5
        syscall
        add  $t0, $t0, $v0 # t0 = t0 + segundo numero
        li   $v0, 1        # syscall 1: imprimir inteiro
        move $a0, $t0
        syscall
        li   $v0, 10       # syscall 10: encerrar
        syscall

Overwriting soma.s


Vamos criar um arquivo com dois valores para serem utilizados no programa acima:

In [50]:
%%writefile entrada.txt
3
4

Overwriting entrada.txt


Agora executamos o comando abaixo:

In [51]:
!java -jar Mars.jar nc sm soma.s < entrada.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
7


Observe o resultado final: 7

## 🎓 Conclusão

Nesta aula, você:
- Instalou e executou o **simulador MARS 4.5** no Google Colab, via linha de comando
- Escreveu seu primeiro programa em assembly MIPS (`hello.s`) e conheceu os segmentos
  `.data` (variáveis) e `.text` (código)
- Entendeu o papel do **terminador nulo**: o "z" do `.asciiz` é o byte 0 que marca
  onde a string acaba — sem ele, a syscall 4 imprimiria lixo da memória
- Passou dados ao programa via **redirecionamento de entrada** (`< entrada.txt`),
  simulando o teclado num ambiente sem interface gráfica

**Conceitos-chave para levar adiante:**
- Registradores são a "mesa de trabalho" da CPU; **syscalls** são a ponte com o
  "sistema operacional" do simulador
- Strings na memória são **bytes**: o endereço em `$a0` basta porque o 0 final
  delimita o fim — o mesmo princípio das strings em C
- O padrão `programa < arquivo` não é gambiarra: é o mecanismo que alimenta
  **casos de teste** e permite correção automática — programa de verdade roda
  com testes, não "no dedo"

**Próximos passos da disciplina:** laços e condicionais (`loop.s`), vetores
(`vetor.s`), funções e pilha (`jal`/`jr $ra`) — e depois abrimos o processador:
datapath **monociclo** e **multiciclo**, para ver como cada instrução percorre
o hardware.